In [1]:
import numpy as np
import pandas as pd 
import os
print(os.listdir("../input"))

['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'description.md', 'jigsaw-toxic-comment-classification-challenge', 'sample_submission.csv.zip', 'train.csv', 'test.csv']


In [2]:
from keras.models import Sequential
from keras.layers import LSTM,Dense,GlobalMaxPool1D,Dropout,Embedding,Bidirectional,Flatten,CuDNNLSTM,Convolution1D,MaxPool1D
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

2026-04-28 02:17:56.115402: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 02:17:56.124993: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777342676.136022      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777342676.139335      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 02:17:56.152496: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ImportError: cannot import name 'CuDNNLSTM' from 'keras.layers' (/usr/local/lib/python3.11/dist-packages/keras/api/layers/__init__.py)

In [3]:
df = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/train.csv')

In [4]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
list_classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
y = df[list_classes].values

In [6]:
tqdm.pandas()

NameError: name 'tqdm' is not defined

Using word embeddings so that words with similar words have similar representation in vector space. It represents every word as a vector. The words which have similar meaning are place close to each other. Quick understanding can be done from [this](https://towardsdatascience.com/word-embeddings-exploration-explanation-and-exploitation-with-code-in-python-5dac99d5d795) article.

In [7]:
f = open('../input/gloveembeddings/glove.6B.50d.txt')
embedding_values = {}
for line in tqdm(f):
    value = line.split(' ')
    word = value[0]
    coef = np.array(value[1:],dtype = 'float32')
    embedding_values[word] = coef

FileNotFoundError: [Errno 2] No such file or directory: '../input/gloveembeddings/glove.6B.50d.txt'

There are many words in the training data which are not there in the glove embeddings. So we take the mean of the embeddings and replace the absent words in the embeddings with mean.

In [8]:
all_embs = np.stack(embedding_values.values())
emb_mean,emb_std = all_embs.mean(), all_embs.std()
emb_mean,emb_std

NameError: name 'embedding_values' is not defined

In [9]:
x = df['comment_text']

These are the steps that needs to be performed so that we can convert each word of our vocabulary into a unique integer. Tokenizer is initalized in first step. Then fitting on the text will help us create a vocabulary so that each word is assigned with a unique integer. Then we convert in the whole sentence of the comment into a sequence of numbers which are assigned by the tokenizer.

In [10]:
token = Tokenizer(num_words=20000)

NameError: name 'Tokenizer' is not defined

In [11]:
token.fit_on_texts(x)

NameError: name 'token' is not defined

In [12]:
seq = token.texts_to_sequences(x)

NameError: name 'token' is not defined

Padding the sequence helps in making all the sentence of same length. maxlen is the parameter which decides the length we want to assign to all the sentences. Padding is done by adding 0 on either the end of sentence or prior the sentence if the sentence is having length less than max length. This is also a parameter which user can change, by defaults its prefix. If the length of the sentence is more than 100 then it is pruned which brings down the length to 50 (maxlen)

In [13]:
pad_seq = pad_sequences(seq,maxlen=50)

NameError: name 'pad_sequences' is not defined

Now we will be converting each word in our vocabulary into word embeddings. This embedding is vector of 1x50 dimension which represents each word as a vector and placing them into a vector space. Embedding matrix is created in which the number assigned to the word by tokenizer is assigned with the corresponding vector which we get from the glove embeddings

In [14]:
vocab_size = len(token.word_index)+1
print(vocab_size)

NameError: name 'token' is not defined

In [15]:
embedding_matrix = np.random.normal(emb_mean, emb_std, (vocab_size, 50))
for word,i in tqdm(token.word_index.items()):
    values = embedding_values.get(word)
    if values is not None:
        embedding_matrix[i] = values

NameError: name 'emb_mean' is not defined

In [16]:
model1 = Sequential()

Here this embedding layer is important as this will help us in training of the sentences with their respective embeddings whihc we have assigned above. The first parameter is the size of our vocabulary. Second parameter is the output embeddings length which is 50 in this case as we used the 50 glove embeddings of each word. The length of each observation which is expected by the network is given by input_length parameter. We have padded all the observations to 50 hence we set input_length = 50. Weights parameter shows that the embeddings which we want to use is embeddings_matrix and it should not be altered hence trainable is kept false. If we want to train our own embeddings we can simply remove the weights and trainable parameter.

In [17]:
model1.add(Embedding(vocab_size,50,input_length=50,weights = [embedding_matrix],trainable = False))

NameError: name 'vocab_size' is not defined

Building a LSTM model. LSTM networks are useful in sequence data as they are capable of remembering the past words which help them in understanding the meaning of the sentence which helps in text classification. Bidirectional Layer is helpful as it helps in understanding thesentence from start to end and also from end to start. It works in both the direction. This is useful as the reverse order LSTM layer is capable of learning patterns which are not possible for the normal LSTM layers which goes from start to end of the sentence in the normal order. Hence Bidirectional layers are useful in text classification problems as different patterns can be captured from 2 directions. CuDNNLSTM is same as LSTM. If you are using GPU then CuDNNLSTM will be faster but if you are using CPU please use LSTM.



In [18]:
model1.add(Bidirectional(CuDNNLSTM(50,return_sequences=True)))
model1.add(GlobalMaxPool1D())

NameError: name 'CuDNNLSTM' is not defined

In [19]:
model1.add(Dense(50,activation = 'relu'))
model1.add(Dropout(0.2))

2026-04-28 02:18:03.287901: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [20]:
model1.add(Dense(6,activation = 'sigmoid'))

In [21]:
model1.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [22]:
history = model1.fit(pad_seq,y,epochs =2,batch_size=32,validation_split=0.1)

NameError: name 'pad_seq' is not defined

In [23]:
model1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
test = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/test.csv')

In [25]:
test.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


Preprocessing of the test data so that model can easily make its prediction as it should be in the same format as that of our training data. Note that we are using the same tokenizer in our testing and we are not fitting it again because this might change the numbers assigned to words which are there in the training data.

In [26]:
x_test = test['comment_text']
test_seq = token.texts_to_sequences(x_test)
test_pad_seq = pad_sequences(test_seq,maxlen=50)

NameError: name 'token' is not defined

In [27]:
predict1 = model1.predict(test_pad_seq)

NameError: name 'test_pad_seq' is not defined

In [28]:
sample_submission = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv')
sample_submission[list_classes] = predict1
sample_submission.to_csv('submission1.csv', index=False)

NameError: name 'predict1' is not defined

In [29]:
model2 = Sequential()
model2.add(Embedding(vocab_size,50,input_length=50,weights = [embedding_matrix],trainable = False))
model2.add(CuDNNLSTM(50))
model2.add(Dense(50,activation = 'relu'))
model2.add(Dense(6,activation = 'sigmoid'))
model2.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])

NameError: name 'vocab_size' is not defined

In [30]:
history = model2.fit(pad_seq,y,epochs =2,batch_size=32,validation_split=0.1)

NameError: name 'pad_seq' is not defined

In [31]:
predict2 = model2.predict(test_pad_seq)
sample_submission = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv')
sample_submission[list_classes] = predict2
sample_submission.to_csv('submission2.csv', index=False)

NameError: name 'test_pad_seq' is not defined

Here we will be using convolution1D model. This model works with a kernel of size 5 which hovers over the sentence capturing only the important features of the model. This will help in extracting the features that are significant to classify the text into different categories. After which we use a MaxPooling layer which will help in taking the maximum value from the kernel size of 2. This will also help us in reducing the computation and extracting only the important feature present under the kernel at that point of time. We can still change the model by changing the number of neurons and also by changing the number of layers which will change the computational complexity of the model too.

In [32]:
model3 = Sequential()
model3.add(Embedding(vocab_size,50,input_length=50,weights = [embedding_matrix],trainable = False))
model3.add(Convolution1D(9,kernel_size=5,activation='relu'))
model3.add(MaxPool1D(2))
model3.add(Flatten())
model3.add(Dense(50,activation = 'relu'))
model3.add(Dense(6,activation = 'sigmoid'))
model3.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])

NameError: name 'vocab_size' is not defined

In [33]:
history = model3.fit(pad_seq,y,epochs =2,batch_size=32,validation_split=0.1)

NameError: name 'pad_seq' is not defined

In [34]:
predict3 = model3.predict(test_pad_seq)
sample_submission = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv')
sample_submission[list_classes] = predict3
sample_submission.to_csv('submission3.csv', index=False)

NameError: name 'test_pad_seq' is not defined

Here we again use convolution1D but changing the number of filters and also the kernel size.

In [35]:
model4 = Sequential()
model4.add(Embedding(vocab_size,50,input_length=50,weights = [embedding_matrix],trainable = False))
model4.add(Convolution1D(18,kernel_size=3,activation='relu'))
model4.add(MaxPool1D(2))
model4.add(Flatten())
model4.add(Dense(50,activation = 'relu'))
model4.add(Dense(6,activation = 'sigmoid'))
model4.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])

NameError: name 'vocab_size' is not defined

In [36]:
history = model4.fit(pad_seq,y,epochs =2,batch_size=32,validation_split=0.1)

NameError: name 'pad_seq' is not defined

In [37]:
predict4 = model4.predict(test_pad_seq)
sample_submission = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv')
sample_submission[list_classes] = predict4
sample_submission.to_csv('submission4.csv', index=False)

NameError: name 'test_pad_seq' is not defined

Making prediction with Ensemble model as well where we can combine all the 4 models that we have trained.

In [38]:
ensemble_prediction = 0.25*predict1+0.25*predict2+0.25*predict3+0.25*predict4
sample_submission = pd.read_csv('../input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv')
sample_submission[list_classes] = ensemble_prediction
sample_submission.to_csv('submission_ensemble.csv', index=False)

NameError: name 'predict1' is not defined